In [2]:
import sys
import os
sys.path.append('/Users/mariana/Documents/projects/Huawei/survan')
sys.path.append('/home/mvargas/code/Huawei/survan')

from lambda_cox import LambdaSA
from utils import get_churn_lastfm_dataset_months, get_targets_and_masks, train_test_split
import yaml
import jax
import jax.numpy as jnp
import haiku as hk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
path = '/Users/mariana/Documents/projects/Huawei/SurvanData/lastfm-dataset-1K'

In [3]:
seqs, ts, cs = get_churn_lastfm_dataset_months(path, 53, use_static_fs=True)

In [4]:
target, h_ws, mask = get_targets_and_masks(seqs, ts, cs, True)

In [5]:
X_train, X_test, y_train, y_test, hws_train, hws_test, \
        m_train, m_test, ts_train, ts_test, cs_train, cs_test = \
train_test_split(seqs, target, h_ws, mask, ts, cs, 32, test_size=0.2)

In [6]:
m_train[ts_train == 5][0][:6, :6]

array([[ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [False, False, False, False, False, False]])

In [8]:
df_events = pd.read_csv(os.path.join(path, 'events.csv'))

In [9]:
df_events.head()

,userid,month,censored,time
0,user_000001,2009-05,0,26
1,user_000002,2009-04,0,38
2,user_000003,2009-05,0,38
3,user_000004,2009-04,0,25
4,user_000005,2009-05,0,32


In [10]:
prof = pd.read_csv(os.path.join(path, 'userid-profile.tsv'), sep='\t')

In [11]:
len(prof)

992

In [21]:
prof.head()

,#id,gender,age,country,registered
0,user_000001,m,25.367133,Japan,"Aug 13, 2006"
1,user_000002,f,25.367133,Peru,"Feb 24, 2006"
2,user_000003,m,22.000000,United States,"Oct 30, 2005"
3,user_000004,f,25.367133,unknown,"Apr 26, 2006"
4,user_000005,m,25.367133,Bulgaria,"Jun 29, 2006"


In [13]:
prof['gender'] = prof.gender.fillna('unknown')

In [18]:
prof['country'] = prof.country.fillna('unknown')

In [20]:
average_age = prof['age'].mean()
prof['age'] = prof['age'].fillna(average_age)

In [27]:
aux = pd.get_dummies(prof, columns=['gender', 'country'])

In [28]:
bool_columns = aux.select_dtypes(include='bool').columns

# Convert boolean columns to float type
aux[bool_columns] = aux[bool_columns].astype(float)

In [1]:
#kkbox dataset
path = '/home/mvargas/code/Huawei/SurvanData/kkbox-churn-prediction-challenge/kkbox'

In [4]:
logs = pd.read_feather(os.path.join(path, 'logs.feather'))
surv = pd.read_feather(os.path.join(path, 'survival_data.feather'))
cov = pd.read_feather(os.path.join(path, 'covariates.feather'))

In [12]:
pd.unique(cov.payment_method_id)

[35, 38, 41, 39, 40, ..., 8, 7, 5, 15, 3]
Length: 39
Categories (39, int64): [2, 3, 4, 5, ..., 38, 39, 40, 41]

In [15]:
cov.head()

,index_survival,msno,n_prev_churns,days_between_subs,days_since_reg_init,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,...,age_at_start,strange_age,nan_days_since_reg_init,no_prev_churns,duration,churn,duration_censor,duration_lcd,churn_lcd,duration_censor_lcd
0,0,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,0,0,4549,35,7,0,0,False,...,28,False,False,True,5,True,142,5,True,142
1,1,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,0,0,1062,38,410,1788,1788,False,...,21,False,False,True,410,True,435,410,True,435
2,2,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,0,0,892,41,30,99,99,True,...,0,True,False,True,119,False,119,74,False,74
3,3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,0,0,1535,39,31,149,149,True,...,24,False,False,True,413,True,729,413,True,729
4,4,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,1,0,2082,39,30,149,149,True,...,25,False,False,False,231,False,231,182,False,182


In [17]:
pd.unique(surv.msno).shape

(2307537,)

In [18]:
len(surv)

2861512

In [19]:
pd.unique(logs.msno).shape

(1368900,)

In [20]:
logs.head()

,msno,semester,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs,...,days_between_subs,start_date,registration_init_time,duration,days_since_reg_init,duration_censor,duration_lcd,churn_lcd,duration_censor_lcd,churn_type
0,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,2016_2,2.016091e+07,4.000000,1.750000,0.750000,1.250000,24.500000,24.750000,7020.406000,...,NaN,2016-09-09,2004-03-27,5,4549.0,142,5,True,142,final
1,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,2015_1,2.015041e+07,1.200000,0.600000,0.466667,0.400000,119.533333,111.400000,28545.502800,...,NaN,2015-11-21,2012-12-24,410,1062.0,435,410,True,435,final
2,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,2015_2,2.015093e+07,0.133333,0.200000,0.133333,0.000000,116.000000,97.066667,28259.872333,...,NaN,2015-11-21,2012-12-24,410,1062.0,435,410,True,435,final
3,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,2016_1,2.016033e+07,1.000000,0.266667,0.400000,0.066667,114.466667,105.933333,28259.763267,...,NaN,2015-11-21,2012-12-24,410,1062.0,435,410,True,435,final
4,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,2016_2,2.016102e+07,2.333333,0.733333,0.400000,0.733333,74.066667,74.266667,18753.343667,...,NaN,2015-11-21,2012-12-24,410,1062.0,435,410,True,435,final


In [23]:
cov.columns

Index(['index_survival', 'msno', 'n_prev_churns', 'days_between_subs',
       'days_since_reg_init', 'payment_method_id', 'payment_plan_days',
       'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'is_cancel',
       'city', 'gender', 'registered_via', 'age_at_start', 'strange_age',
       'nan_days_since_reg_init', 'no_prev_churns', 'duration', 'churn',
       'duration_censor', 'duration_lcd', 'churn_lcd', 'duration_censor_lcd'],
      dtype='object')

In [27]:
cols = ['msno', 'days_between_subs',
       'days_since_reg_init', 'payment_method_id', 'payment_plan_days',
       'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'is_cancel',
       'city', 'gender', 'registered_via', 'age_at_start', 'strange_age',
       'nan_days_since_reg_init']

In [28]:
cov = cov[cols]

In [39]:
pd.unique(logs.msno).shape

(1368900,)

In [38]:
pd.unique(cov_oh.msno).shape

(2307489,)

In [32]:
cov_oh = pd.get_dummies(cov, columns=['city', 'gender', 'payment_method_id', 'registered_via'])

In [35]:
# Select boolean columns
bool_columns = cov_oh.select_dtypes(include='bool').columns

# Convert boolean columns to float
cov_oh[bool_columns] = cov_oh[bool_columns].astype(float)

In [40]:
path

'/home/mvargas/code/Huawei/SurvanData/kkbox-churn-prediction-challenge/kkbox'

In [41]:
cov_oh.to_feather(os.path.join(path, 'covariates_onehot.feather'))

In [43]:
cov_oh.columns

Index(['msno', 'days_between_subs', 'days_since_reg_init', 'payment_plan_days',
       'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'is_cancel',
       'age_at_start', 'strange_age', 'nan_days_since_reg_init', 'city_1',
       'city_3', 'city_4', 'city_5', 'city_6', 'city_7', 'city_8', 'city_9',
       'city_10', 'city_11', 'city_12', 'city_13', 'city_14', 'city_15',
       'city_16', 'city_17', 'city_18', 'city_19', 'city_20', 'city_21',
       'city_22', 'gender_female', 'gender_male', 'payment_method_id_2',
       'payment_method_id_3', 'payment_method_id_4', 'payment_method_id_5',
       'payment_method_id_6', 'payment_method_id_7', 'payment_method_id_8',
       'payment_method_id_10', 'payment_method_id_11', 'payment_method_id_12',
       'payment_method_id_13', 'payment_method_id_14', 'payment_method_id_15',
       'payment_method_id_16', 'payment_method_id_17', 'payment_method_id_18',
       'payment_method_id_19', 'payment_method_id_20', 'payment_method_id_21',
   